# React 框架 · 第 12 天

## React + TypeScript：极客园移动端项目

**资料依据**：`React+TS极客园移动端.pdf`（15 页）、`项目初始化 / 列表模块 / 详情模块` 3 篇 MD 笔记，以及 `react-jike-mobile` 示例代码。压缩包中的 `(1)` 文件与同名原件重复，本笔记只按原件整理。课程项目采用 React 18、TypeScript 5、Vite 4、React Router 6、Ant Design Mobile 5、Axios 1。

**项目链路**：搭建工程 → 配置路由与别名 → 封装请求和类型 → 获取频道 → 按频道展示文章 → 上拉加载更多 → 跳转详情并展示文章。

## 一、本日目标与页面结构

完成本日后应能：

1. 初始化 React + TS 移动端工程，配置 Ant Design Mobile、路由和 `@` 别名。
2. 把 Axios 的响应包裹结构与各接口的业务数据结构组合成泛型类型。
3. 让频道 `Tabs` 驱动不同文章列表，并正确传递 `channelId`。
4. 理解 `pre_timestamp` 游标式加载更多、列表拼接与 `hasMore` 停止条件。
5. 将文章 ID 传给详情页，处理参数缺失、加载中与请求失败。

![课程资料中的首页组件关系](assets/day12/3.png)

首页 `Home` 负责频道导航；每个 `HomeList` 负责对应频道的文章和分页。详情页 `Detail` 从 URL 读取文章 ID，再请求并展示文章。组件、API 类型和请求逻辑各有边界，排错时可沿数据链逐层检查。

## 二、工程初始化与依赖

课程创建命令：

```bash
npm create vite@latest react-jike-mobile -- --template react-ts
cd react-jike-mobile
npm install
npm install antd-mobile react-router-dom axios normalize.css
npm install -D @types/node
npm run dev
```

资料代码的实际依赖版本见 `package.json`；复现课程时以配套项目版本为准。入口 `main.tsx` 导入 `normalize.css`，然后用 `RouterProvider` 渲染路由树。配套 `App.tsx` 只是组件库测试代码，最终入口直接使用路由，并未渲染那个测试组件。

```text
src/
  apis/       接口请求与响应类型
  pages/Home/ 首页、频道、自定义 Hook、文章列表
  pages/Detail/ 文章详情
  router/     路由表
  utils/      Axios 实例及统一导出
  main.tsx    应用入口
```

## 三、建立首页与详情页路由

```tsx
// src/router/index.tsx
import { createBrowserRouter } from 'react-router-dom'
import Home from '@/pages/Home'
import Detail from '@/pages/Detail'

export const router = createBrowserRouter([
  { path: '/', element: <Home /> },
  { path: '/detail', element: <Detail /> },
])
```

```tsx
// src/main.tsx
import ReactDOM from 'react-dom/client'
import { RouterProvider } from 'react-router-dom'
import 'normalize.css'
import { router } from './router'

ReactDOM.createRoot(document.getElementById('root')!).render(
  <RouterProvider router={router} />
)
```

`/` 展示频道文章列表，`/detail` 展示文章详情。路由 URL 承担页面定位和文章 ID 传递，刷新详情页后也能重新读到 ID。`index.html` 应保留移动端 `viewport` 配置，使页面按设备宽度布局。

## 四、`@` 别名必须同时让构建工具和 TS 认识

课程用 `@` 表示 `src`。Vite 配置负责实际解析；TS 配置负责类型检查和编辑器提示。两边只配一边，常出现“编辑器不报错但运行失败”或反过来的情况。

```ts
// vite.config.ts（课程配置写法）
import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'
import path from 'path'

export default defineConfig({
  plugins: [react()],
  resolve: { alias: { '@': path.resolve(__dirname, './src') } },
})
```

```jsonc
// tsconfig.json 的 compilerOptions 中
{
  "baseUrl": ".",
  "paths": { "@/*": ["src/*"] }
}
```

配好后才能稳定使用 `import { http } from '@/utils'`、`import Home from '@/pages/Home'` 等路径。以上对应课程中的 Vite 4 / TS 5 项目配置，迁移到其他模板时应检查其配置文件布局。

## 五、封装 Axios：公共配置集中，接口放到 API 模块

```ts
// src/utils/http.ts
import axios from 'axios'

export const httpInstance = axios.create({
  baseURL: 'http://geek.itheima.net/v1_0',
  timeout: 5000,
})

httpInstance.interceptors.request.use(config => config)
httpInstance.interceptors.response.use(response => response)

// src/utils/index.ts
export { httpInstance as http } from './http'
```

课程先建立实例，再留出请求/响应拦截器入口；当前示例只是原样放行。以后若增加 token、统一错误提示，可在这里集中处理，但应保证返回值结构与 API 类型一致。

**特别注意两层 `data`**：Axios 返回的 `response.data` 是接口响应体；课程接口响应体里又有 `data` 字段，所以频道列表位于 `response.data.data.channels`。没有改变响应拦截器返回值前，不能直接写成 `response.data.channels`。

## 六、泛型响应类型：公共外壳 + 具体业务数据

```ts
// src/apis/shared.ts
export type ResType<T> = {
  message: string
  data: T
}

// src/apis/list.ts
import { http } from '@/utils'
import type { ResType } from './shared'

export type ChannelItem = { id: number; name: string }
type ChannelRes = { channels: ChannelItem[] }

export function fetchChannelAPI() {
  return http.request<ResType<ChannelRes>>({ url: '/channels' })
}
```

`http.request<ResType<ChannelRes>>` 使 `response.data` 的类型成为 `ResType<ChannelRes>`，继续访问 `response.data.data.channels` 时得到 `ChannelItem[]`。

![课程资料中的 Axios 泛型响应结构](assets/day12/2.png)

**资料核对**：MD 的某一段把频道 `id` 写成 `string`，配套 API 代码写成 `number`。这份笔记跟随配套代码，在页面传文章接口参数时显式 `String(item.id)`。TS 类型只描述预期形状，不会自动检查服务器实际返回的 JSON。

## 七、频道 Tabs：把获取逻辑提取为自定义 Hook

```tsx
// src/pages/Home/useTabs.ts
import { useEffect, useState } from 'react'
import { fetchChannelAPI } from '@/apis/list'
import type { ChannelItem } from '@/apis/list'

export function useTabs() {
  const [channels, setChannels] = useState<ChannelItem[]>([])
  const [error, setError] = useState('')

  useEffect(() => {
    let active = true
    fetchChannelAPI()
      .then(res => {
        if (active) setChannels(res.data.data.channels)
      })
      .catch(() => {
        if (active) setError('频道加载失败')
      })
    return () => { active = false }
  }, [])

  return { channels, error }
}
```

自定义 Hook 复用的是状态与副作用逻辑，不会自动使所有调用它的组件共享同一份状态。课程最简例在 `catch` 中直接 `throw`；在异步 Effect 内这样做不利于展示错误，页面更适合保存错误状态并提供重试入口。

## 八、用真实频道渲染导航，并传递 `channelId`

```tsx
import { Tabs } from 'antd-mobile'
import HomeList from './HomeList'
import { useTabs } from './useTabs'
import './style.css'

function Home() {
  const { channels, error } = useTabs()
  if (error) return <p>{error}</p>

  return (
    <div className="tabContainer">
      <Tabs defaultActiveKey="0">
        {channels.map(channel => (
          <Tabs.Tab title={channel.name} key={channel.id}>
            <div className="listContainer">
              <HomeList channelId={String(channel.id)} />
            </div>
          </Tabs.Tab>
        ))}
      </Tabs>
    </div>
  )
}
```

`Tabs.Tab` 的 `key` 标识频道项；`HomeList` 的 `channelId` 是请求参数，不应在列表内部写死成 `'0'`。`defaultActiveKey="0"` 依赖频道数据中确有对应 key；若频道集合不含 0，应改成首个实际频道的 key 或受控激活值。课程展示的是推荐频道作为初始项的情形。

![课程资料中的频道导航示意](assets/day12/4.png)

## 九、滚动容器：无限加载需要真实的可滚动区域

```css
.tabContainer {
  position: fixed;
  top: 0;
  height: 50px;
  width: 100%;
}

.listContainer {
  position: fixed;
  top: 50px;
  bottom: 0;
  width: 100%;
  overflow: auto;
}
```

课件将 Tabs 放在顶部，将文章列表放在独立滚动容器中。若忘记 `overflow: auto`、容器没有高度，或 `InfiniteScroll` 放在错误的层级，滑动到底部时可能不会按预期触发。移动端布局要同时在窄屏上检查顶部频道是否遮住内容。

## 十、文章列表接口：参数与响应类型

```ts
export type ListItem = {
  art_id: string
  title: string
  pubdate: string
  aut_name: string
  cover: { type: number; images: string[] }
}

export type ListRes = {
  results: ListItem[]
  pre_timestamp: string
}

export type ListParams = {
  channel_id: string
  timestamp: string
}

export function fetchListAPI(params: ListParams) {
  return http.request<ResType<ListRes>>({
    url: '/articles',
    params,
  })
}
```

课程完整 `ListItem` 还包含作者 ID、评论数、置顶标记等字段；上例只保留列表展示用的字段。`timestamp` 是请求文章列表的游标，服务端返回的 `pre_timestamp` 用于下一次加载。这里不是常见的 `page=2` 页码模式，不要自行把它当页数加一。

## 十一、首屏列表：频道变化时重新获取

```tsx
type Props = { channelId: string }

function HomeList({ channelId }: Props) {
  const [rows, setRows] = useState<ListItem[]>([])
  const [cursor, setCursor] = useState(() => String(Date.now()))
  const [hasMore, setHasMore] = useState(true)
  const [ready, setReady] = useState(false)
  const [error, setError] = useState('')

  useEffect(() => {
    let active = true
    setRows([])
    setHasMore(true)
    setReady(false)
    setError('')

    fetchListAPI({ channel_id: channelId, timestamp: String(Date.now()) })
      .then(res => {
        if (!active) return
        const page = res.data.data
        setRows(page.results)
        setCursor(page.pre_timestamp)
        setHasMore(page.results.length > 0)
        setReady(true)
      })
      .catch(() => {
        if (!active) return
        setError('文章加载失败')
        setHasMore(false)
        setReady(true)
      })

    return () => { active = false }
  }, [channelId])

  // 列表渲染与加载更多见下节
}
```

核心点是依赖 `[channelId]`：频道变化时清空旧列表、重置分页，再用新频道参数请求。`active` 避免旧请求晚返回后覆盖新频道的数据。错误时关闭无限加载并显示 `error`；实际页面还应提供手动重试入口。

## 十二、渲染文章行：ID、标题、封面与时间

```tsx
import { Image, List } from 'antd-mobile'
import { useNavigate } from 'react-router-dom'

function ArticleRows({ rows }: { rows: ListItem[] }) {
  const navigate = useNavigate()
  return (
    <List>
      {rows.map(article => (
        <List.Item
          key={article.art_id}
          onClick={() => navigate(`/detail?id=${encodeURIComponent(article.art_id)}`)}
          prefix={article.cover.images[0]
            ? <Image src={article.cover.images[0]} width={40} height={40} fit="cover" />
            : null}
          description={article.pubdate}
        >
          {article.title}
        </List.Item>
      ))}
    </List>
  )
}
```

`HomeList` 可渲染 `<ArticleRows rows={rows} />`。`key` 用稳定的 `art_id`；封面图片可能为空数组，先判断再渲染；点击行时带文章 ID 跳转。课程前期先用 `users` 假数据验证 `List` / `Image` 结构，再切换到接口数据，这种“先布局、后接数据”的顺序便于定位问题。

## 十三、上拉加载：游标、拼接、停止条件

```tsx
const loadingMoreRef = useRef(false)

const loadMore = async () => {
  if (!ready || !hasMore || loadingMoreRef.current) return
  loadingMoreRef.current = true
  try {
    const res = await fetchListAPI({
      channel_id: channelId,
      timestamp: cursor,
    })
    const page = res.data.data
    setRows(oldRows => [...oldRows, ...page.results])
    setCursor(page.pre_timestamp)
    setHasMore(page.results.length > 0)
  } finally {
    loadingMoreRef.current = false
  }
}

// 与上节的 <List> 放在同一滚动区域
{ready && <InfiniteScroll loadMore={loadMore} hasMore={hasMore} />}
```

`InfiniteScroll` 到达底部后调用 `loadMore`；本次传入上次响应的 `pre_timestamp`，把新 `results` 拼到旧列表末尾。空页时把 `hasMore` 设为 `false`，组件停止继续请求。函数式 `setRows(oldRows => ...)` 可避免直接使用旧闭包中的数组；`loadingMoreRef` 阻止同一时刻重复加载。

**实际项目还需补上**：`catch` 错误状态与重试，以及频道切换时忽略旧频道的分页响应。课件示例在失败时直接 `throw`，且用 `[...]listRes.results` 拼接，容易造成未处理错误或并发下的旧状态问题。这里展示的是主流程与修正方向，不是整页可直接粘贴的完整组件。

## 十四、详情页路由参数与 API 类型

课程用查询参数传文章 ID：`/detail?id=文章ID`。详情页用 `useSearchParams` 读取，结果类型是 `string | null`，因为用户可能直接打开缺少 `id` 的 URL。

```ts
// src/apis/detail.ts
export type DetailDataType = {
  art_id: string
  title: string
  pubdate: string
  content: string
  aut_name: string
}

export function fetchDetailAPI(id: string) {
  return http.request<ResType<DetailDataType>>({
    url: `/articles/${encodeURIComponent(id)}`,
  })
}
```

与列表类型一样，上例只保留此处需要的主要字段；配套代码还定义了点赞、收藏、作者头像等更多详情字段。路由参数可能为空，应在发请求前判断，不应直接用 `id!` 消除 TS 提醒。

## 十五、详情页：加载中、成功、失败和返回

```tsx
import { useEffect, useState } from 'react'
import { useNavigate, useSearchParams } from 'react-router-dom'
import { NavBar } from 'antd-mobile'
import { fetchDetailAPI } from '@/apis/detail'
import type { DetailDataType } from '@/apis/detail'

function Detail() {
  const navigate = useNavigate()
  const [params] = useSearchParams()
  const id = params.get('id')
  const [detail, setDetail] = useState<DetailDataType | null>(null)
  const [error, setError] = useState('')

  useEffect(() => {
    if (!id) return
    let active = true
    setDetail(null)
    setError('')
    fetchDetailAPI(id)
      .then(res => { if (active) setDetail(res.data.data) })
      .catch(() => { if (active) setError('文章加载失败') })
    return () => { active = false }
  }, [id])

  if (!id) return <p>缺少文章 ID</p>
  if (error) return <p>{error}</p>
  if (!detail) return <p>正在加载文章…</p>

  return (
    <div>
      <NavBar onBack={() => navigate(-1)}>{detail.title}</NavBar>
      <p>{detail.pubdate} · {detail.aut_name}</p>
      {/* 正文 HTML 的处理见下一节 */}
    </div>
  )
}
```

页面分别处理了缺参、请求失败、加载中和成功四种状态。`active` 可防止路由快速切换时旧请求的结果覆盖新文章。生产项目还可在卸载时取消网络请求，避免无用流量。

## 十六、文章正文是 HTML：注意渲染边界

课件使用：

```tsx
<div dangerouslySetInnerHTML={{ __html: detail.content }} />
```

这是因为接口中的 `content` 是一段 HTML 字符串；直接写 `{detail.content}` 只会把标签当普通文字显示。`dangerouslySetInnerHTML` 会让浏览器解析该 HTML，也就要求内容来源可信。若正文可能包含不可信输入，先经过可靠的 HTML 清理流程，并仅允许业务需要的标签与属性；不要把任意用户输入直接插入 DOM。

课程此处重点是页面数据流，配套示例未展示清理环节。阅读或复用代码时应把它视为待补的边界，而不是默认认为服务端内容一定安全。

## 十七、把完整数据流连起来

```text
Home 首次加载
  └─ useTabs → GET /channels → ChannelItem[] → Tabs
         └─ 某个频道 id → HomeList(channelId)
                ├─ GET /articles?channel_id=...&timestamp=...
                ├─ results → 文章行；pre_timestamp → 下一次游标
                └─ InfiniteScroll → 取下一批 → 拼接结果
                       └─ 点击 art_id → /detail?id=...
                              └─ GET /articles/:id → Detail
```

调试时按链路检查：**URL/参数 → 请求 → 响应外壳 → `data` 里的业务数据 → 状态 → 组件**。类型通常从 API 模块流向状态和 Props；若在组件里大量写 `any` 或强制断言，说明接口边界需要重新确认。

## 十八、资料中的易错点与修正

1. **频道 ID 类型不一致**：笔记一段写 `string`，配套代码是 `number`；传给 `channel_id: string` 时显式转换。
2. **频道参数被写死**：早期列表例子固定 `channel_id: '0'`；最终应使用 `channelId` Prop。
3. **两层 `data` 混淆**：Axios `res.data` 是接口响应，业务内容位于 `res.data.data`。
4. **无限滚动没有滚动盒子**：确认容器高度、`overflow: auto` 和 `InfiniteScroll` 所在位置。
5. **切频道后保留旧列表/游标**：频道变化时重置，并防止旧请求覆盖新结果。
6. **并发加载用旧数组拼接**：用函数式更新，并用加载中标志阻止重复请求。
7. **在异步请求 `catch` 中直接 `throw`**：页面不会自动得到友好的错误界面；改为保存错误并提供重试。
8. **`id!` 掩盖路由缺参**：先判断 `id`，再调用详情接口。
9. **直接插入任意 HTML**：只渲染可信或经过清理的正文。

## 十九、练习与验收清单

1. 访问首页能看到真实频道；切换频道后请求参数变化，列表内容随之更新。
2. 空封面文章仍可正常显示标题；列表项点击后，地址栏包含对应文章 ID。
3. 滑动到底部会请求下一批；旧数据保留、新数据追加；没有更多数据时停止加载。
4. 快速切换频道时不出现上一个频道的数据；请求失败时有明确提示。
5. 直接打开 `/detail` 会显示缺参提示；打开带合法 `id` 的详情页能看到标题与正文；返回按钮能回到列表。
6. 执行 `npm run build`，检查 TS 类型错误与资源路径问题。接口能否实时访问取决于课程服务的当前状态，必要时可用课程提供的响应样例或本地假数据验证组件逻辑。

## 二十、本日总结

第 12 天把前面学到的 React、TypeScript、路由和 Hooks 串进一个移动端项目：类型从接口泛型进入状态与组件，频道 ID 决定文章列表，`pre_timestamp` 驱动下一批加载，文章 ID 决定详情。最重要的工程习惯是让**参数、响应类型、状态和页面展示保持一致**，并把空值、失败和并发切换这些真实场景纳入实现。